# Submit SFT + DPO training as HF Job

Runs on HF Jobs A10g GPU (~$1.50/hr). Trains Qwen 3B with SFT (1 epoch) + DPO (3 epochs), evals, uploads everything to a HF model repo. Total ~$5-8 of $30 credit.

Setup before running: have your **HF token** with **write** scope ready. Get from https://huggingface.co/settings/tokens — make sure it's a **fine-grained token with WRITE permission** to your repos.

Run cells top-to-bottom.

In [ ]:
# Cell 1 — install hf hub + cli
!pip install -q -U huggingface_hub

In [ ]:
# Cell 2 — auth. Paste your HF token (write scope) when prompted.
from huggingface_hub import login, HfApi
import getpass
HF_TOKEN = getpass.getpass('HF token (write scope): ')
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi()
me = api.whoami()
USERNAME = me['name']
print(f'authed as: {USERNAME}')

In [ ]:
# Cell 3 — submit the job.
# Switch JOB_TYPE to choose:
#   "sft_dpo"     = single SFT+DPO pipeline (~$3-4, fast)
#   "coevolution" = 3-round agent+adversary co-evolution (~$8-12, deep)
import subprocess

JOB_TYPE = 'coevolution'   # 'sft_dpo' OR 'coevolution'

if JOB_TYPE == 'sft_dpo':
    SCRIPT_URL = 'https://raw.githubusercontent.com/SAISriram19/meta/main/scripts/hfjobs_train.py'
    HUB_REPO_ID = f'{USERNAME}/meta-qwen3b-sft-dpo'
    EXTRA_ENVS = ['-e', 'SFT_EPOCHS=1', '-e', 'DPO_EPOCHS=3']
else:  # coevolution
    SCRIPT_URL = 'https://raw.githubusercontent.com/SAISriram19/meta/main/scripts/hfjobs_coevolution.py'
    HUB_REPO_ID = f'{USERNAME}/meta-coevolution'
    EXTRA_ENVS = []  # coevolution uses defaults

MODEL = 'Qwen/Qwen2.5-3B-Instruct'
SCENARIOS = 'L0_launch,L2_strategic_shift,L4_market_pivot'

# Pre-create the destination repo
api.create_repo(repo_id=HUB_REPO_ID, repo_type='model', exist_ok=True)
print(f'job_type:    {JOB_TYPE}')
print(f'script:      {SCRIPT_URL}')
print(f'destination: https://huggingface.co/{HUB_REPO_ID}')

# Coevolution needs more VRAM (two LoRAs trained sequentially) → a10g-large.
flavor = 'a10g-large' if JOB_TYPE == 'coevolution' else 'a10g-small'
cmd = [
    'hf', 'jobs', 'uv', 'run',
    '--flavor', flavor,
    '--with', 'transformers',
    '--with', 'peft',
    '--with', 'trl',
    '--with', 'bitsandbytes',
    '--with', 'datasets',
    '--with', 'accelerate',
    '--with', 'matplotlib',
    '--with', 'pyyaml',
    '--with', 'networkx',
    '--with', 'pydantic',
    '--with', 'huggingface_hub',
    '-s', f'HF_TOKEN={HF_TOKEN}',
    '-e', f'HUB_REPO_ID={HUB_REPO_ID}',
    '-e', f'AGENT_MODEL={MODEL}',
    '-e', f'ADVERSARY_MODEL={MODEL}',
    '-e', f'MODEL={MODEL}',
    '-e', f'SCENARIOS={SCENARIOS}',
    *EXTRA_ENVS,
    SCRIPT_URL,
]
print('\nsubmitting:')
print(' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print('--- stdout ---'); print(result.stdout)
print('--- stderr ---'); print(result.stderr)

In [ ]:
# Cell 4 — list your jobs (find the one we just submitted)
import subprocess
r = subprocess.run(['hf', 'jobs', 'ps'], capture_output=True, text=True)
print(r.stdout)
print(r.stderr)

In [ ]:
# Cell 5 — stream logs of the most recent job. Replace JOB_ID if needed.
import subprocess
JOB_ID = ''  # paste from Cell 4 output if you need a specific one; empty = latest
if JOB_ID:
    cmd = ['hf', 'jobs', 'logs', JOB_ID, '--follow']
else:
    # heuristic: take the first line of `hf jobs ps`
    ps = subprocess.run(['hf', 'jobs', 'ps'], capture_output=True, text=True).stdout.strip().splitlines()
    if len(ps) > 1:
        JOB_ID = ps[1].split()[0]
        print(f'streaming logs for {JOB_ID}')
        cmd = ['hf', 'jobs', 'logs', JOB_ID, '--follow']
    else:
        cmd = None
        print('no jobs found')
if cmd:
    subprocess.run(cmd)

In [ ]:
# Cell 6 — pull results once job finishes. Reads from the HF model repo we created.
from huggingface_hub import snapshot_download
import json

local = snapshot_download(repo_id=HUB_REPO_ID, repo_type='model')
print(f'downloaded to {local}')

import os
for f in sorted(os.listdir(local)):
    sz = os.path.getsize(os.path.join(local, f)) if os.path.isfile(os.path.join(local, f)) else '-'
    print(f'  {f}  {sz}')

# Show eval results inline
ev_path = os.path.join(local, 'eval_results.json')
if os.path.exists(ev_path):
    print('\n--- eval_results.json ---')
    print(json.dumps(json.load(open(ev_path)), indent=2))

# Show training curve
from IPython.display import Image, display
img_path = os.path.join(local, 'training_curves.png')
if os.path.exists(img_path):
    display(Image(img_path))